In [ ]:
# @title Imports y Param
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import  LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

NORMALIZAR=1


In [ ]:
#@title Cargar Dataset y preprocesar
# Cargar el dataset del Titanic (desde archivo local)
dataset_path = 'titanic.csv' # Asegurate de que el archivo 'titanic.csv' esté subido a tu sesión de Colab
df = pd.read_csv(dataset_path)


# ------------ Comienza el EDA  ------------------------------
# Cantidad de registros y columnas
print(f"Cantidad de registros: {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")

print('--- Exploratory Data Analysis (EDA) ---')

# Cantidad de valores nulos por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

#  Distribution of 'Survived'
plt.figure(figsize=(6, 4))
sns.countplot(x='Survived', data=df, palette='viridis', hue='Survived', legend=False)
plt.title('Distribución de Survived (0 = No, 1 = Yes)')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.show()

#  Survival by 'Sex'
plt.figure(figsize=(7, 5))
sns.barplot(x='Sex', y='Survived', data=df, palette='magma', hue='Sex', legend=False)
plt.title('Tasa de supervivencia por sexo (0 = Male, 1 = Female)')
plt.xlabel('Sex')
plt.ylabel('Survival Rate')
plt.xticks(ticks=[0, 1], labels=['Male', 'Female'])
plt.show()

#  Survival by 'Pclass'
plt.figure(figsize=(7, 5))
sns.barplot(x='Pclass', y='Survived', data=df, palette='crest', hue='Pclass', legend=False)
plt.title('Tasa de supervivencia por clase de pasajero')
plt.xlabel('Passenger Class')
plt.ylabel('Survival Rate')
plt.show()

# Conteo por clase de pasajero y supervivencia (Bar plot agrupado con Plotly Express)
fig_pclass_survived = px.histogram(df, x='Pclass', color='Survived', barmode='group',
                                   title='Conteo por Clase de Pasajero y Supervivencia',
                                   labels={'Pclass': 'Clase de Pasajero', 'Survived': 'Sobrevivió'})
fig_pclass_survived.update_layout(xaxis=dict(tickmode='linear')) # Asegura que las Pclass se muestren como 1, 2, 3
fig_pclass_survived.show()

#  Distribution of 'Age' with respect to 'Survived'
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Age', hue='Survived', kde=True, palette='coolwarm', multiple='dodge')
plt.title('Distribución por edades según la supervivencia')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

#  Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación')
plt.show()

print('--- End of EDA ---')
# ---------  FIN EDA ------------------------------------------------

# ------------ Preparación de los datos -----------------------------

# Eliminamos columnas con muchos nulos y las que no usaremos
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df  = df.drop(columns=columns_to_drop)
df= df.dropna()

# Convertir variables categóricas a numéricas
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked')#  one-hot columna 'Embarked'

# Separar características (X) y variable objetivo (y)
X = df.drop('Survived', axis=1)
y = df['Survived']
column_names= X.columns

# Dividir Train/Test
porc_test= 0.2
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size= porc_test)
print("\t %d datos para training" %x_train.shape[0])
print("\t %d datos para testing" %x_test.shape[0])

#normalizar Datos
from sklearn.preprocessing import StandardScaler
if (NORMALIZAR):
    scaler = StandardScaler()
    scaler.fit(x_train)
    x_train = scaler.transform(x_train)
    x_test = scaler.transform(x_test)
# vuelve a crear dataframes (solo con fines visuales y prácticos)
x_train = pd.DataFrame(x_train, columns=column_names)
x_test = pd.DataFrame(x_test, columns=column_names)

In [ ]:
# @title Clasificación con KNN

from sklearn.neighbors import KNeighborsClassifier

K=5
modelo = KNeighborsClassifier(n_neighbors=K)

modelo.fit(x_train,y_train)

# métricas
y_pred_train= modelo.predict(x_train)
acc_train= accuracy_score(y_train, y_pred_train)
print(f"TRAIN {acc_train:.2f}")

y_pred_test= modelo.predict(x_test)
acc_test= accuracy_score(y_test, y_pred_test)
print(f"TEST {acc_test:.2f}")

In [ ]:
# @title cross validation
from sklearn.model_selection import ShuffleSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np


# definimos el modelo
K=5
modelo = KNeighborsClassifier(n_neighbors=K)

# Creamos el Pipeline que primero normaliza y luego aplica el modelo
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Primer paso: Normalización
    ('mlp', modelo)            # Segundo paso: El modelo MLP
])

# Define el número de divisiones aleatorias que quieres realizar
N_SPLITS_RANDOM = 10

# Define el tamaño del conjunto de prueba en cada división
TEST_SIZE_PROPORTION = 0.2

# Crea el objeto ShuffleSplit
ss = ShuffleSplit(n_splits=N_SPLITS_RANDOM, test_size=TEST_SIZE_PROPORTION)

# Define la única métrica que queramos evaluar
scoring = ['accuracy']

# Ejecuta la validación cruzada con ShuffleSplit
# Pasamos el objeto ss a cv en cross_validate
cv_results_shuffle = cross_validate(pipeline, X, y, cv=ss, scoring=scoring, return_train_score=True)

print("\n--- Resultados promedio y desviación estándar ---")
# Imprime los resultados promediados y la std para accuracy
mean_accuracy_train = cv_results_shuffle['train_accuracy'].mean()
std_accuracy_train = cv_results_shuffle['train_accuracy'].std()
mean_accuracy_test = cv_results_shuffle['test_accuracy'].mean()
std_accuracy_test = cv_results_shuffle['test_accuracy'].std()
print(f"Accuracy TRAIN: {mean_accuracy_train:.2f} (+/- {std_accuracy_train:.2f})")
print(f"Accuracy TEST: {mean_accuracy_test:.2f} (+/- {std_accuracy_test:.2f})")

# Prepare data for plotting  each split
train_scores = cv_results_shuffle['train_accuracy']
test_scores = cv_results_shuffle['test_accuracy']

split_numbers = [f'Split {i+1}' for i in range(N_SPLITS_RANDOM)]
# Create DataFrame for plotting
plot_data = {
    'Split': split_numbers * 2,
    'Metric Value': list(train_scores) + list(test_scores),
    'Metric': ['Acc (Train)'] * N_SPLITS_RANDOM + ['Acc (Test)'] * N_SPLITS_RANDOM,
    'Set': ['Train'] * N_SPLITS_RANDOM + ['Test'] * N_SPLITS_RANDOM
}
plot_df = pd.DataFrame(plot_data)

# Get the colors from the viridis palette used by seaborn for 'Acc (Train)' and 'Acc (Test)'
palette = sns.color_palette('viridis', n_colors=2)
train_color = palette[0]; test_color = palette[1]

# Create bar plot using Seaborn
plt.figure(figsize=(12, 6))
sns.barplot(x='Split', y='Metric Value', hue='Metric', data=plot_df, palette='viridis')
plt.axhline(mean_accuracy_train, color=train_color, linestyle='--', label=f'Mean Train Acc: {mean_accuracy_train:.2f}')
plt.axhline(mean_accuracy_test, color=test_color, linestyle='--', label=f'Mean Test Acc: {mean_accuracy_test:.2f}')
plt.title('Accuracy Scores Across Cross-Validation Splits')
plt.xlabel('Cross-Validation Split')
plt.ylabel('Accuracy')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Metric & Mean Accuracy')
plt.tight_layout()
plt.show()
